# Python Properties and Descriptors — Interview Preparation

Covers: @property, getter/setter/deleter, data vs non-data descriptors, __set_name__, cached_property, and interview Q&A.

## 1. @property — Getter, Setter, Deleter

In [ ]:
class Temperature:
    def __init__(self, celsius=0):
        self._celsius = celsius  # _ prefix = private by convention

    @property
    def celsius(self):
        """Getter — accessed like an attribute"""
        return self._celsius

    @celsius.setter
    def celsius(self, value):
        """Setter — validates before storing"""
        if value < -273.15:
            raise ValueError(f'Temperature below absolute zero: {value}')
        self._celsius = value

    @celsius.deleter
    def celsius(self):
        print('Deleting temperature')
        del self._celsius

    @property
    def fahrenheit(self):
        return self._celsius * 9/5 + 32

    @fahrenheit.setter
    def fahrenheit(self, value):
        self.celsius = (value - 32) * 5/9  # reuses celsius setter (with validation)


t = Temperature(25)
print('celsius:', t.celsius)       # getter
print('fahrenheit:', t.fahrenheit) # computed property

t.celsius = 100                    # setter
print('After setting 100°C, fahrenheit:', t.fahrenheit)

t.fahrenheit = 32                  # fahrenheit setter → celsius setter
print('After setting 32°F, celsius:', t.celsius)

try:
    t.celsius = -300               # ValueError
except ValueError as e:
    print('Error:', e)

del t.celsius                      # deleter

## 2. Data vs Non-Data Descriptors

In [ ]:
class DataDescriptor:
    """Data descriptor: has __get__ AND __set__ — takes priority over instance __dict__"""
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return 'data descriptor value'

    def __set__(self, obj, value):
        print(f'DataDescriptor.__set__ called with {value}')


class NonDataDescriptor:
    """Non-data descriptor: only __get__ — instance __dict__ takes priority"""
    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return 'non-data descriptor value'


class MyClass:
    data = DataDescriptor()
    nondata = NonDataDescriptor()


obj = MyClass()
obj.__dict__['data'] = 'instance value'
obj.__dict__['nondata'] = 'instance value'

print('obj.data:', obj.data)       # 'data descriptor value' — descriptor wins!
print('obj.nondata:', obj.nondata) # 'instance value' — instance dict wins!

## 3. Reusable Validation Descriptor

In [ ]:
class Validated:
    """Base data descriptor for validated attributes"""
    def __set_name__(self, owner, name):
        """Called automatically when assigned to a class attribute"""
        self.name = name
        self.storage_name = f'_{name}'

    def __get__(self, obj, objtype=None):
        if obj is None:
            return self
        return getattr(obj, self.storage_name, None)

    def __set__(self, obj, value):
        self.validate(value)
        setattr(obj, self.storage_name, value)

    def validate(self, value):
        pass  # override in subclasses


class PositiveNumber(Validated):
    def validate(self, value):
        if not isinstance(value, (int, float)) or value <= 0:
            raise ValueError(f'{self.name} must be a positive number, got {value!r}')


class NonEmptyString(Validated):
    def validate(self, value):
        if not isinstance(value, str) or not value.strip():
            raise ValueError(f'{self.name} must be a non-empty string')


class Product:
    name = NonEmptyString()    # descriptor assigned here — __set_name__ called
    price = PositiveNumber()
    quantity = PositiveNumber()

    def __init__(self, name, price, quantity):
        self.name = name
        self.price = price
        self.quantity = quantity

    @property
    def total_value(self):
        return self.price * self.quantity

    def __repr__(self):
        return f'Product({self.name!r}, ${self.price}, qty={self.quantity})'


p = Product('Widget', 9.99, 100)
print(p)
print('Total value:', p.total_value)

try:
    p.price = -5
except ValueError as e:
    print('Error:', e)

try:
    p.name = ''
except ValueError as e:
    print('Error:', e)

## 4. cached_property

In [ ]:
from functools import cached_property

class DataProcessor:
    def __init__(self, data):
        self.data = data

    @cached_property
    def sorted_data(self):
        """Computed once, cached in instance __dict__"""
        print('Computing sorted_data...')
        return sorted(self.data)

    @cached_property
    def statistics(self):
        print('Computing statistics...')
        data = self.sorted_data
        n = len(data)
        return {
            'min': data[0],
            'max': data[-1],
            'mean': sum(data) / n,
            'median': data[n // 2],
        }


dp = DataProcessor([3, 1, 4, 1, 5, 9, 2, 6])
print('First access:')
print(dp.sorted_data)   # computes
print('Second access:')
print(dp.sorted_data)   # cached — no print!
print('Statistics:', dp.statistics)

# Cached in instance __dict__
print('In __dict__:', 'sorted_data' in dp.__dict__)

## 5. Interview Practice

In [ ]:
# Q: Make an attribute read-only
class ImmutablePoint:
    def __init__(self, x, y):
        self._x = x
        self._y = y

    @property
    def x(self):
        return self._x

    @property
    def y(self):
        return self._y
    # No setters defined — read-only!

    def __repr__(self):
        return f'ImmutablePoint({self._x}, {self._y})'


p = ImmutablePoint(3, 4)
print('x:', p.x, 'y:', p.y)

try:
    p.x = 10
except AttributeError as e:
    print('Error:', e)

In [ ]:
# Q: property is itself a descriptor — demonstrate
class MyClass:
    @property
    def value(self):
        return 42

# property is a data descriptor stored in the class __dict__
print('type of MyClass.value:', type(MyClass.value))  # <class 'property'>
print('has __get__:', hasattr(MyClass.value, '__get__'))
print('has __set__:', hasattr(MyClass.value, '__set__'))

# Accessing on instance calls __get__
obj = MyClass()
print('obj.value:', obj.value)  # 42